In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import pickle
from torch.utils.data import Subset
from pathlib import Path

from data.datasets import dataset_from_schema
from model.loader import load_model, load_thresholds
from model.model import MultitaskMLPWithQRegression
from model.predict import run_inference, predict_single
from predicates.interface import PredictionInterface, PredictionEntry, ContinuousRange
from predicates.compiler import compile_asp_constraint

In [3]:
checkpoint_path = "../models/latest/multitask_mlp-epoch13.best.pt"
checkpoint_path = "../models/latest/multitask_mlp-epoch13.best.pt"

In [4]:
# Load the model and reference dataset
model, ref_dataset, device = load_model(checkpoint_path)
print(f"Loaded model {type(model)}")

Loaded model <class 'model.model.MultitaskMLPWithQRegression'>


In [5]:
thresholds = load_thresholds(Path(checkpoint_path).resolve())
if thresholds is not None:
    print("Conformal thresholds loaded.")
else:
    print("No conformal thresholds found. Run calibrate.py on this checkpoint.")

No conformal thresholds found. Run calibrate.py on this checkpoint.


In [18]:
from data.datasets import RawFeatureDataset

# Load test dataset for inference
with open("../data/splits/synthetic-ei/test.pkl", "rb") as f:
    test_idx = pickle.load(f)
test_subset = Subset(ref_dataset, test_idx)
test_ds = RawFeatureDataset.from_subset(test_subset)
test_ds.normalise_continuous(ref_dataset._cont_mean.tolist(), ref_dataset._cont_std.tolist());

In [25]:
import numpy as np
# Compare underlying raw records to avoid ambiguous tensor boolean checks
a = test_ds._load_record(0)
b = test_subset.dataset._load_record(test_subset.indices[0])

assert a["smiles"] == b["smiles"], "SMILES strings do not match"
assert np.allclose(a["structural_vector"], b["structural_vector"]), "Structural vectors do not match"
assert np.allclose(a["feature_vector"], b["feature_vector"]), "Feature vectors do not match"
assert a["structural_features"] == b["structural_features"], "Structural features do not match"

In [26]:
import numpy as np
from tqdm import tqdm

smallest_formula_space = np.inf
smallest_idx = None
for idx in tqdm(range(min(len(test_ds), 1000))):
    x, _ = test_ds[idx]
    pred = predict_single(x, model, test_ds, thresholds=thresholds)
    formula_space = np.prod([hi - lo + 1 for lo, hi in pred.count_ranges.values()])
    if formula_space < smallest_formula_space:
        smallest_formula_space = formula_space
        smallest_idx = idx

print(f"Smallest formula space: {smallest_formula_space}")
print(f"Index of smallest formula space: {smallest_idx}")

  0%|          | 0/1000 [00:00<?, ?it/s]

  5%|▍         | 47/1000 [00:03<01:02, 15.18it/s]


KeyboardInterrupt: 

In [ ]:
h_count_idx = ref_dataset.continuous_feature_order.index("element_H")
ref_dataset._cont_mean.tolist()

[0.24891303479671478,
 0.007613104302436113,
 1.0578029155731201,
 0.7734912037849426,
 0.36984071135520935,
 0.20882543921470642,
 11.854487419128418,
 1.3843673467636108,
 1.1548633575439453,
 0.11595960706472397,
 2.462607153574936e-05,
 1.698254942893982,
 2.1795477867126465,
 0.11351980268955231,
 0.8671925067901611,
 8.532365798950195,
 0.13443008065223694,
 0.09852435439825058,
 1.2683184146881104,
 1.0,
 4.564842700958252,
 13.992654800415039,
 8.064289093017578,
 0.4680759608745575,
 5.408292770385742,
 2.0334577560424805,
 0.0,
 0.0,
 20.518362045288086,
 0.0,
 0.0,
 24.4953556060791,
 0.0,
 2.8212411403656006,
 3.3662662506103516,
 0.02122858539223671,
 0.36277851462364197,
 0.0,
 0.251365602016449,
 0.9770503044128418,
 376.5137634277344,
 0.01266509760171175,
 0.9762440323829651,
 0.6078106760978699,
 0.0,
 27.089872360229492,
 6.57151460647583,
 4.312119960784912,
 2.259394645690918,
 0.03069411776959896,
 0.1364722102880478,
 0.1636384129524231,
 0.032767269760370255,
 3

In [27]:
idx = 720
x, gt = test_ds[idx]
pred = predict_single(x, model, test_ds, thresholds=thresholds)
print(pred)

RecordPrediction
  counts:
    num_carbons: [19, 27]
    num_hydrogens: [21, 38]
    num_nitrogens: [0, 3]
    num_oxygens: [3, 8]
    num_phosphorus: 0
    num_sulfurs: [0, 1]
  continuous (83 features):
    alcohol: [-0.01, 2.43]
    aldehyde: 0
    aliphatic_ring_count: [0.02, 5.37]
    amide: [-0.00, 1.62]
    amine: [-0.00, 1.00]
    aniline: 0
    aromatic_atom_count: [1.38, 14.72]
    aromatic_heteroatom_count: [0.00, 2.21]
    aromatic_n_count: [-0.00, 2.02]
    aromatic_o_count: [0.00, 1.02]
    aromatic_p_count: 0
    aromatic_ring: [0.14, 2.37]
    aromatic_ring_count: [0.34, 2.82]
    aromatic_s_count: 0
    assigned_chiral_center_count: [-0.01, 7.87]
    branch_point_count: [7.04, 14.50]
    bridgehead_atom_count: [-0.00, 3.30]
    carboxylic_acid: [-0.00, 1.04]
    chiral_center_count: [-0.04, 8.43]
    connected_component_count: 1
    degree_1_atom_count: [3.25, 10.15]
    degree_2_atom_count: [7.78, 18.25]
    degree_3_atom_count: [5.70, 13.97]
    degree_4plus_atom_cou

In [16]:
# Print the unnormalized ground truth atom counts
from model.predict import _ELEMENT_TO_COUNT_KEY

record = test_ds._load_record(0)
feats = record["structural_features"]
atom_counts_gt = {}
for feat, key in _ELEMENT_TO_COUNT_KEY.items():
    atom_counts_gt[key] = int(feats.get(feat, 0))
print("Ground truth atom counts:")
for key, count in atom_counts_gt.items():
    print(f"{key}: {count}")
smiles = record["smiles"]
print(f"SMILES: {smiles}")

Ground truth atom counts:
num_carbons: 16
num_hydrogens: 20
num_nitrogens: 4
num_oxygens: 6
num_phosphorus: 0
num_sulfurs: 0
SMILES: CC(C)(C)OC(=O)C1CCC/C1=N/Nc1ccc([N+](=O)[O-])cc1[N+](=O)[O-]


In [9]:
# Convert into a prediction interface object
interface = pred.to_interface()
for key, value in interface.root.items():
    print(f"{key}: {value}")

1: pred_set=[0] alpha=0.05 proba=[0.9773359447717667]
2: pred_set=[0] alpha=0.05 proba=[0.9766263626515865]
3: pred_set=[0] alpha=0.05 proba=[0.9801546093076468]
4: pred_set=[0] alpha=0.05 proba=[0.9807805996388197]
5: pred_set=[0] alpha=0.05 proba=[0.9793272353708744]
6: pred_set=[0] alpha=0.05 proba=[0.9798414222896099]
7: pred_set=[0] alpha=0.05 proba=[0.9806574191898108]
8: pred_set=[0] alpha=0.05 proba=[0.8896719291806221]
9: pred_set=[0] alpha=0.05 proba=[0.9794865790754557]
10: pred_set=[0] alpha=0.05 proba=[0.9788464698940516]
11: pred_set=[0] alpha=0.05 proba=[0.8685432076454163]
12: pred_set=[0] alpha=0.05 proba=[0.9786617252975702]
13: pred_set=[0] alpha=0.05 proba=[0.86042520403862]
14: pred_set=[0] alpha=0.05 proba=[0.6543902158737183]
15: pred_set=[0] alpha=0.05 proba=[0.9350842162966728]
16: pred_set=[0] alpha=0.05 proba=[0.9269342571496964]
17: pred_set=[0] alpha=0.05 proba=[0.8960140496492386]
18: pred_set=[0] alpha=0.05 proba=[0.9779432695358992]
19: pred_set=[0] alph

In [10]:
asp_constraints, atom_counts = compile_asp_constraint(interface)
for c in asp_constraints:
    print(c)
print(atom_counts)


        halogenated_count(N) :-
            element_F_count(F),
            element_Cl_count(Cl),
            element_Br_count(Br),
            element_I_count(I),
            N = F + Cl + Br + I.
    

        ring_span_size(CY, N) :-
            cycle_start(S, CY),
            cycle_end(E, CY),
            N = E - S + 1.
    

        % ring_path_atom(CY, I): I lies on the backbone DFS path from
        % cycle_start(CY) to cycle_end(CY).  Replaces the old span heuristic
        % I2 - I1 + 1 which overcounts branch atoms.
        ring_path_atom(CY, I) :-
            cycle_start(S, CY),
            cycle_end(E, CY),
            is_descendant(S, I),
            is_descendant(I, E).
    

        ring_size_3_count(N) :-
            N = #count { CYCLE : ring_size(CYCLE, 3) }.
    

            has_n_o_bond :-
                any_bond_between(I, J),
                symbol(I, "N"),
                symbol(J, "O").
        

            has_oc_nc :-
                any_bond_between(I, J),


In [ ]:
# Run ASP
from asp.bridge import solve

# Override atom counts
print(f"Solving for C{atom_counts_gt['num_carbons']}, H{atom_counts_gt['num_hydrogens']}, N{atom_counts_gt['num_nitrogens']}, O{atom_counts_gt['num_oxygens']}, S{atom_counts_gt['num_sulfurs']}")
%time candidates, _, _= solve(asp_constraints, atom_counts_gt, output_list_of_smiles=True, timeout=10)

print(f"Found {len(candidates)} candidate molecules.")
for smi in candidates:
    print(smi)

Solving for C6, H12, N2, O0, S2


In [11]:
# Molecular formula combos
n = 1
for lo, hi in pred.count_ranges.values():
    n *= (hi - lo + 1)
print(n, "combinations")

41400 combinations


In [12]:
import itertools

all_candidates = []
seen = set()

combos = itertools.product(*[range(lo, hi + 1) for lo, hi in pred.count_ranges.values()])
for combo in tqdm(combos):
    counts = dict(zip(pred.count_ranges, combo))
    interface = pred.to_interface(counts=counts)
    asp_constraints, atom_counts = compile_asp_constraint(interface)
    candidates, _, _ = solve(asp_constraints, atom_counts, output_list_of_smiles=True)
    for smi in candidates:
        if smi not in seen:
            seen.add(smi)
            all_candidates.append(smi)
print(f"{len(all_candidates)} unique candidates across all formulas")

0it [00:00, ?it/s]


NameError: name 'solve' is not defined